# Result Property Descriptions (TITLE / UNIT) from .MX1

Every ``.MX1`` file embeds, for each result-property channel it defines, a human-readable ``TITLE`` (German description) and a ``UNIT`` (physical unit) - metadata not otherwise exposed via the Toolkit API or the plain SQLite model schema (see `DPKT` for the SQL-queryable equivalent, which is result-properties-only, same as this). This notebook opens each of the three network-type models that turned out to be load-bearing for the object-types RST reference (see `object_types.ipynb` - Steam turned out to add nothing not already covered by the other three), inserts every object type possible into each, runs a calculation so a real `.MX1` gets written, and builds a combined `{OBJTYPE~ATTRTYPE: {TITLE, UNIT, DATATYPE}}` map from all three - meant to be merged with the object-types RST reference to add a description/unit column for every result property.

## Import/Init

In [1]:
from sir3stoolkit.core import wrapper

In [2]:
wrapper.Initialize_Toolkit(r"C:\3S\SIR 3S\SirGraf-90-15-00-24_Quebec-Upd2")

[2026-09-03 15:57:16,778] INFO in sir3stoolkit.core.wrapper: [Initialization] Using provided SirGraf path: C:\3S\SIR 3S\SirGraf-90-15-00-24_Quebec-Upd2
[2026-09-03 15:57:16,778] INFO in sir3stoolkit.core.wrapper: [Initialization] Using provided SirGraf path: C:\3S\SIR 3S\SirGraf-90-15-00-24_Quebec-Upd2
[2026-09-03 15:57:16,836] INFO in sir3stoolkit.core.wrapper: [Initialization] Initializing toolkit with SirGraf path: C:\3S\SIR 3S\SirGraf-90-15-00-24_Quebec-Upd2


In [3]:
from sir3stoolkit.mantle.dataframes import SIR3S_Model_Dataframes

In [4]:
s3s = SIR3S_Model_Dataframes()

[2026-09-03 15:57:29,336] INFO in sir3stoolkit.core.wrapper: [Model Class Initialization] Initialization complete


In [5]:
import xml.etree.ElementTree as ET
import json
from pathlib import Path

## Configuration

In [6]:
network_files = [
    ("Water", "./result_props_Water.db3"),
    ("Gas", "./result_props_Gas.db3"),
    ("DistrictHeating", "./result_props_DistrictHeating.db3"),
]

output_dir = (Path.cwd().resolve() / "./output/.").resolve()
output_dir.mkdir(parents=True, exist_ok=True)
network_files

[('Water', './result_props_Water.db3'),
 ('Gas', './result_props_Gas.db3'),
 ('DistrictHeating', './result_props_DistrictHeating.db3')]

## 1) Per network: open, insert every possible object type, calculate, read `.MX1`

Inserting every `ObjectTypes` member with a bare `InsertElement(type, "-1")` (no endpoints/container/position wiring) is expected to fail for many types (connecting elements need two nodes, table rows need a parent, etc.) - each attempt is wrapped so a failure just means that type is skipped, not that the run aborts. The existing base topology in each `result_props_<Network>.db3` is what lets the calculation actually run afterward.

In [7]:
def insert_all_object_types(s3s):
    inserted, failed = [], []
    s3s.StartTransaction("Insert every possible object type")
    for i in range(0, 150):
        try:
            object_type = s3s.ObjectTypes(i)
        except Exception:
            continue
        try:
            tk = s3s.InsertElement(object_type, "-1")
            if tk and tk != "-1":
                inserted.append(object_type.name)
            else:
                failed.append(object_type.name)
        except Exception:
            failed.append(object_type.name)
    s3s.EndTransaction()
    return inserted, failed

In [8]:
def read_mx1_descriptions(s3s):
    mx1_path = s3s.get_current_mx1_filepath()
    if not Path(mx1_path).exists():
        print(f"  MX1 not found at {mx1_path}")
        return {}
    root = ET.parse(mx1_path).getroot()
    descriptions = {}
    for ch in root.findall("XL1"):
        objtype = ch.get("OBJTYPE")
        attrtype = ch.get("ATTRTYPE")
        if not objtype or not attrtype:
            continue
        key = f"{objtype}~{attrtype}"
        descriptions[key] = {
            "OBJTYPE": objtype,
            "ATTRTYPE": attrtype,
            "TITLE": ch.get("TITLE"),
            "UNIT": ch.get("UNIT"),
            "DATATYPE": ch.get("DATATYPE"),
        }
    return descriptions

In [9]:
s3s.AllowSirMessageBox(bAllow=False)  # avoid blocking popups during unattended batch insertion
s3s.EnableOrDisableOutputComments(outputComments=False)  # ~150 InsertElement calls would otherwise spam output

In [10]:
all_descriptions = {}
per_network_summary = {}

for network_name, db_path in network_files:
    print(f"=== {network_name} ===")
    s3s.OpenModel(dbName=db_path, providerType=s3s.ProviderTypes.SQLite, Mid="M-1-0-1",
                  saveCurrentlyOpenModel=False, namedInstance="", userID="", password="")

    inserted, failed = insert_all_object_types(s3s)
    print(f"  inserted {len(inserted)} object types, {len(failed)} not insertable this way")
    s3s.SaveChanges()

    try:
        s3s.ExecCalculation(waitForSirCalcToExit=True)
    except Exception as e:
        print(f"  ExecCalculation raised: {e}")

    net_descriptions = read_mx1_descriptions(s3s)
    print(f"  {len(net_descriptions)} result-property channels described in .MX1")
    per_network_summary[network_name] = len(net_descriptions)

    for key, val in net_descriptions.items():
        all_descriptions.setdefault(key, val)

    s3s.CloseModel(saveChangesBeforeClosing=False)

per_network_summary, len(all_descriptions)

=== Water ===


[2026-09-03 15:57:39,902] ERROR in sir3stoolkit.core.wrapper: Error : DistrictHeatingConsumer: Element Type cannot be inserted into this Network Type
[2026-09-03 15:57:39,903] ERROR in sir3stoolkit.core.wrapper: Error : DistrictHeatingFeeder: Element Type cannot be inserted into this Network Type
[2026-09-03 15:57:40,001] ERROR in sir3stoolkit.core.wrapper: Error : Compressor: Element Type cannot be inserted into this Network Type
[2026-09-03 15:57:40,003] ERROR in sir3stoolkit.core.wrapper: Error : HeaterCooler: Element Type cannot be inserted into this Network Type
[2026-09-03 15:57:40,379] ERROR in sir3stoolkit.core.wrapper: Error : HeatExchanger: Element Type cannot be inserted into this Network Type
[2026-09-03 15:57:40,381] ERROR in sir3stoolkit.core.wrapper: Error : HeatFeederConsumerStation: Element Type cannot be inserted into this Network Type
[2026-09-03 15:57:40,685] ERROR in sir3stoolkit.core.wrapper: Error : Repository of Element not found.


  inserted 141 object types, 7 not insertable this way
  555 result-property channels described in .MX1
=== Gas ===


[2026-09-03 15:57:59,753] ERROR in sir3stoolkit.core.wrapper: Error : SafetyValve: Element Type cannot be inserted into this Network Type
[2026-09-03 15:57:59,755] ERROR in sir3stoolkit.core.wrapper: Error : PressureRegulator: Element Type cannot be inserted into this Network Type
[2026-09-03 15:57:59,757] ERROR in sir3stoolkit.core.wrapper: Error : DifferentialRegulator: Element Type cannot be inserted into this Network Type
[2026-09-03 15:57:59,758] ERROR in sir3stoolkit.core.wrapper: Error : FlapValve: Element Type cannot be inserted into this Network Type
[2026-09-03 15:57:59,759] ERROR in sir3stoolkit.core.wrapper: Error : PhaseSeparation: Element Type cannot be inserted into this Network Type
[2026-09-03 15:57:59,761] ERROR in sir3stoolkit.core.wrapper: Error : FlowControlUnit: Element Type cannot be inserted into this Network Type
[2026-09-03 15:57:59,764] ERROR in sir3stoolkit.core.wrapper: Error : Pump: Element Type cannot be inserted into this Network Type
[2026-09-03 15:57:5

  inserted 129 object types, 19 not insertable this way
  321 result-property channels described in .MX1
=== DistrictHeating ===


[2026-09-03 15:58:15,449] ERROR in sir3stoolkit.core.wrapper: Error : PhaseSeparation: Element Type cannot be inserted into this Network Type
[2026-09-03 15:58:15,535] ERROR in sir3stoolkit.core.wrapper: Error : VentValve: Element Type cannot be inserted into this Network Type
[2026-09-03 15:58:15,540] ERROR in sir3stoolkit.core.wrapper: Error : StandPipe: Element Type cannot be inserted into this Network Type
[2026-09-03 15:58:15,542] ERROR in sir3stoolkit.core.wrapper: Error : VentilatedPressureAirVessel: Element Type cannot be inserted into this Network Type
[2026-09-03 15:58:15,542] ERROR in sir3stoolkit.core.wrapper: Error : Compressor: Element Type cannot be inserted into this Network Type
[2026-09-03 15:58:15,545] ERROR in sir3stoolkit.core.wrapper: Error : HeaterCooler: Element Type cannot be inserted into this Network Type
[2026-09-03 15:58:15,584] ERROR in sir3stoolkit.core.wrapper: Error : Hydrant: Element Type cannot be inserted into this Network Type
[2026-09-03 15:58:15,7

  inserted 140 object types, 8 not insertable this way
  633 result-property channels described in .MX1


({'Water': 555, 'Gas': 321, 'DistrictHeating': 633}, 752)

## Manual additions: FS / DSI / DSK

`FS`, `DSI`, `DSK` appear as result properties (via `GetResultProperties_from_elementType()`) on 8 connecting-element types (`DPRG`, `KLAP`, `MREG`, `PREG`, `REGV`, `ROHR`, `SIVE`, `VENT`) but never showed up in any `.MX1` this notebook read - not even Tutorial051's model, a real, successfully-calculated network (checked separately, outside this notebook). They're also absent from `DPKT`, the calc engine's own registered-datapoint catalog. Whatever SIR 3S setting/condition emits them, none of the three test networks here (nor Tutorial051) trigger it, so they can't be sourced automatically the way everything else in this map is. Added by hand from domain knowledge instead:

- `FS`: dynamische axiale Last
- `DSI`: dynamische Stützkraft Knoten I
- `DSK`: dynamische Stützkraft Knoten K
- Unit for all three: `[N|kN|MN]`

Applied identically to all 8 object types below, since they always co-occur together (same 8 types for all three properties) - strong signal they're gated by the same single feature/flag.

In [11]:
MANUAL_ENTRIES = {
    "TITLE": {
        "FS": "dynamische axiale Last",
        "DSI": "dynamische St\u00fctzkraft Knoten I",
        "DSK": "dynamische St\u00fctzkraft Knoten K",
    },
    "UNIT": "[N|kN|MN]",
}
MANUAL_OBJTYPES = ["DPRG", "KLAP", "MREG", "PREG", "REGV", "ROHR", "SIVE", "VENT"]

manual_added = 0
for objtype in MANUAL_OBJTYPES:
    for attrtype, title in MANUAL_ENTRIES["TITLE"].items():
        key = f"{objtype}~{attrtype}"
        all_descriptions[key] = {
            "OBJTYPE": objtype,
            "ATTRTYPE": attrtype,
            "TITLE": title,
            "UNIT": MANUAL_ENTRIES["UNIT"],
            "DATATYPE": None,
            "SOURCE": "manual",
        }
        manual_added += 1

print(f"Added {manual_added} manual entries. Total descriptions: {len(all_descriptions)}")

Added 24 manual entries. Total descriptions: 776


## 2) Save the combined map

In [12]:
out_path = output_dir / "result_property_descriptions.json"
with open(out_path, "w", encoding="utf-8") as f:
    json.dump(all_descriptions, f, indent=2, ensure_ascii=False)
print(f"Wrote {len(all_descriptions)} descriptions to {out_path}")

Wrote 776 descriptions to C:\Users\aUsername\3S\sir3stoolkit\docs\source\tutorials\SIR3S_Model_Mantle\TutorialTest_Assets\object_types\output\result_property_descriptions.json


In [13]:
dict(list(all_descriptions.items())[:5])

{'ALLG~TIMESTAMP': {'OBJTYPE': 'ALLG',
  'ATTRTYPE': 'TIMESTAMP',
  'TITLE': 'Zeitstempel nach ISO 8601',
  'UNIT': '[text]',
  'DATATYPE': 'CHAR'},
 'ALLG~SNAPSHOTTYPE': {'OBJTYPE': 'ALLG',
  'ATTRTYPE': 'SNAPSHOTTYPE',
  'TITLE': 'Typ des Zeitpunktes/Ausgabedatensatzes',
  'UNIT': '[text]',
  'DATATYPE': 'CHAR'},
 'ALLG~CVERSO': {'OBJTYPE': 'ALLG',
  'ATTRTYPE': 'CVERSO',
  'TITLE': 'Versionskennung',
  'UNIT': '[text]',
  'DATATYPE': 'CHAR'},
 'ALLG~EXSTAT': {'OBJTYPE': 'ALLG',
  'ATTRTYPE': 'EXSTAT',
  'TITLE': 'Exit-Status der Berechnung',
  'UNIT': '[]',
  'DATATYPE': 'INT4'},
 'ALLG~NFEHL': {'OBJTYPE': 'ALLG',
  'ATTRTYPE': 'NFEHL',
  'TITLE': 'Anzahl Fehler im Berechnungsabschnitt',
  'UNIT': '[]',
  'DATATYPE': 'INT4'}}